In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from tqdm import tqdm
import random

def train_transformer_model(model, train_data, val_data=None, num_epochs=30, batch_size=128, learning_rate=0.1*(1/256), device='cuda'):

    save_path = 'model_history/best_model.pth'
    model.to(device)

    x_train, y_train = train_data
    x_train_f, x_val, y_train_f, y_val = train_test_split(x_train, y_train, test_size=0.01, random_state=42)

    train_loader = DataLoader(TensorDataset(x_train_f, y_train_f), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=batch_size, shuffle=False)
    
    # 손실함수: 목적지 예측 오차 (사용자 정의 DestinationLoss)
    # 참고 논문: "Attention Is All You Need" (arXiv:1706.03762) 기반 Transformer 학습    
    criterion = DestinationLoss(weight_main=0.9)

    # 옵티마이저: AdamW (Weight Decay 분리) 사용
    # 참고 논문: "Decoupled Weight Decay Regularization" (arXiv:1711.05101)
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.00025)

    # 학습률 스케줄러: Cosine Annealing with Warm Restarts
    # 참고 논문: "SGDR: Stochastic Gradient Descent with Warm Restarts" (arXiv:1608.03983)
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=num_epochs, T_mult=max(1, num_epochs//10))

    best_val_loss = float('inf')

    for epoch in range(1, num_epochs + 1):
        torch.cuda.empty_cache()
        model.train()
        total_loss = 0.0

        # Teacher Forcing 비율 점진적 감소
        # 참고 논문: "Scheduled Sampling" (arXiv:1506.03099)
        teacher_forcing_ratio = max(0.0, 1.0 - epoch / num_epochs)  # ⬅️ Gradual decay

        loop = tqdm(train_loader, desc=f"[Epoch {epoch}/{num_epochs}] Train", leave=False)
        for batch_x, batch_y in loop:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            # -------------------------------
            # 입력 데이터에 미세한 노이즈 추가 (regularization)
            # 참고 논문: "Understanding Deep Learning Requires Rethinking Generalization" (arXiv:1611.03530)
            # -------------------------------
            with torch.no_grad():
                feature_noise_std = torch.tensor([4e-4] * batch_x.size(-1), device=batch_x.device)
                noise = torch.randn_like(batch_x) * feature_noise_std
                batch_x += noise

            optimizer.zero_grad()

            # -------------------------------
            # Scheduled Sampling 기반 Autoregressive Training
            # (모델의 예측값을 다음 입력에 섞어 넣음)
            # 참고 논문: "Scheduled Sampling for Sequence Prediction with Recurrent Neural Networks" (arXiv:1506.03099)
            # -------------------------------
            B, T, F = batch_x.shape
            output_len, output_dim = 1, batch_y.shape[-1]

            seq_in = batch_x.clone()
            all_preds = []
            
            for t in range(output_len):  # currently 1 step
                out = model(seq_in)                  # shape: (B, 1, 4)
                pred = out[:, -1, :]                 # (B, 4)
                all_preds.append(pred.unsqueeze(1))  

                use_pred = torch.rand(B, device=device) > teacher_forcing_ratio

                # Only used if output_len > 1 in the future
                if t < output_len - 1:
                    pred_input = pred.detach()
                    true_input = batch_y[:, t, :]     # (B, 4)

                    mixed_input = torch.where(use_pred.unsqueeze(1), pred_input, true_input)  # (B, 4)

                    # Build new input feature for next time
                    dest_lat, dest_lon = batch_x[:, -1, 4], batch_x[:, -1, 5]
                    delta_lat = dest_lat - mixed_input[:, 0]
                    delta_lon = dest_lon - mixed_input[:, 1]
                    dist = torch.sqrt(delta_lat**2 + delta_lon**2)

                    next_input = torch.cat([
                        mixed_input,                                 # 4
                        dest_lat.unsqueeze(1),                       # 1
                        dest_lon.unsqueeze(1),                       # 1
                        dist.unsqueeze(1),                           # 1
                    ], dim=1)                                        # → (B, 7)

                    seq_in = torch.cat([seq_in[:, 1:, :], next_input.unsqueeze(1)], dim=1)

            # Final loss
            preds = torch.cat(all_preds, dim=1)  # (B, 1, 4)
            loss = criterion(preds, batch_y)
            loss.backward()

            # -------------------------------
            # Gradient Clipping (기울기 폭주 방지)
            # 참고 논문: "Attention Is All You Need" (arXiv:1706.03762)
            # -------------------------------
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()
            loop.set_postfix(loss=loss.item())

        avg_loss = total_loss / len(train_loader)
        print(f"[Epoch {epoch}/{num_epochs}] Train Loss: {avg_loss:.6f}")

        # -------------------------------
        # 검증 (Validation) 단계 - Scheduled Sampling 없이 직접 예측만 수행
        # -------------------------------
        model.eval()
        total_val_loss = 0.0
        with torch.no_grad():
            for val_x, val_y in val_loader:
                val_x, val_y = val_x.to(device), val_y.to(device)
                val_out = model(val_x)
                val_loss = criterion(val_out, val_y)
                total_val_loss += val_loss.item()

        avg_val_loss = total_val_loss / len(val_loader)
        print(f"           ↳ Val Loss: {avg_val_loss:.6f}")

        # -------------------------------
        # 베스트 모델 저장 (Validation loss 기준)
        # -------------------------------
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model, save_path)
            print(f"           ↳ ✅ Best model saved (Val Loss: {best_val_loss:.6f})")

        scheduler.step()

for i in range(10):
    model_name = f"model_{i}.pth"
    
    # 모델 생성
    model = TransformerPredictor(input_size=7, output_size=4)
    
    # numpy → torch tensor로 변환
    input_tensor = torch.tensor(input_seqs, dtype=torch.float32)
    output_tensor = torch.tensor(output_seqs, dtype=torch.float32)
    # 학습
    train_transformer_model(model, (input_tensor, output_tensor), num_epochs=100, device='cuda' if torch.cuda.is_available() else 'cpu')

    torch.save(model, model_name)